# Подготовка данных для ручной валидации intent-разметки DialogSum-RU

## Цель блокнота

Этот блокнот готовит **сбалансированную выборку** для **ручной проверки** weak-label разметки намерений (intent) на уровне реплик, полученной в блокноте [`09_intent_modeling_dialogsum_ru.ipynb`](09_intent_modeling_dialogsum_ru.ipynb).

## Зачем нужна ручная валидация

В блокноте 09 разметка intent-классов выполнена **слабыми правилами (weak supervision)** на основе лексических и синтаксических эвристик. Такая разметка содержит шум, поэтому для адекватной оценки baseline-моделей и анализа ошибок нужен **золотой подмножество (gold subset)**, размеченное вручную.

## Что делает этот блокнот

1. Загружает weak-label датасет реплик из `results/tables/dialogsum_ru_intent_utterances_weak.{parquet,csv}`.
2. Формирует **сбалансированную выборку (~500 реплик)** по 15 intent-классам, сохраняя редкие классы.
3. Добавляет вспомогательные поля для удобной ручной разметки.
4. Сохраняет файлы в `data/annotation/` (CSV для людей, Parquet для последующей обработки).
5. Создаёт **codebook** с инструкцией для аннотатора и описанием 15 классов.

Никакие модели здесь не обучаются — это **подготовительный шаг** перед ручной разметкой.

## Ячейка 1. Импорты и настройки

In [ ]:
# ячейка 1: импорты и настройки
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd

# Воспроизводимость
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Удобное отображение таблиц
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

print("Импорты и настройки готовы. SEED =", SEED)

## Ячейка 2. Пути проекта в Google Drive

При запуске в Google Colab сначала смонтируйте Drive:

```python
from google.colab import drive
drive.mount('/content/drive')
```

In [ ]:
# ячейка 2: пути проекта
BASE_DIR = Path("/content/drive/MyDrive/russian-dialogue-intent-thesis")
TABLES_DIR = BASE_DIR / "results" / "tables"
FIGURES_DIR = BASE_DIR / "results" / "figures"
ANNOTATION_DIR = BASE_DIR / "data" / "annotation"

# Создаём папку для файлов ручной валидации (если её ещё нет)
ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

# Ожидаемые входные файлы
WEAK_PARQUET = TABLES_DIR / "dialogsum_ru_intent_utterances_weak.parquet"
WEAK_CSV = TABLES_DIR / "dialogsum_ru_intent_utterances_weak.csv"
TOPIC_CLUSTERS_PARQUET = TABLES_DIR / "dialogsum_ru_topic_clusters_sota.parquet"

# Выходные файлы
OUT_CSV = ANNOTATION_DIR / "dialogsum_ru_intent_manual_review.csv"
OUT_PARQUET = ANNOTATION_DIR / "dialogsum_ru_intent_manual_review.parquet"
OUT_CODEBOOK = ANNOTATION_DIR / "dialogsum_ru_intent_manual_review_codebook.md"

print("BASE_DIR        :", BASE_DIR)
print("TABLES_DIR      :", TABLES_DIR)
print("ANNOTATION_DIR  :", ANNOTATION_DIR)
print("WEAK_PARQUET    :", WEAK_PARQUET)
print("WEAK_CSV        :", WEAK_CSV)
print("TOPIC_CLUSTERS  :", TOPIC_CLUSTERS_PARQUET)
print()
print("OUT_CSV         :", OUT_CSV)
print("OUT_PARQUET     :", OUT_PARQUET)
print("OUT_CODEBOOK    :", OUT_CODEBOOK)

## Схема интентов и краткая инструкция для аннотатора

Каждой реплике приписан **один** intent-класс — тип речевого акта говорящего. Это **utterance-level speech-act intent**, а не тема разговора.

### 15 классов намерений

| Класс | Описание |
|---|---|
| `greeting` | Приветствие |
| `thanks` | Благодарность |
| `farewell` | Завершение разговора |
| `informational_request` | Запрос информации (что/где/когда/сколько/...) |
| `clarification_request` | Уточняющий вопрос по уже сказанному |
| `service_request` | Просьба выполнить действие/услугу |
| `purchase_or_booking_request` | Покупка, заказ или бронирование |
| `complaint` | Жалоба, выражение недовольства |
| `problem_report` | Сообщение о проблеме/неисправности/инциденте |
| `arrangement` | Договорённость о времени/месте/условиях |
| `confirmation` | Подтверждение, согласие |
| `rejection` | Отказ, несогласие |
| `suggestion_or_recommendation` | Совет, рекомендация, предложение |
| `opinion_or_preference` | Мнение, оценка, предпочтение |
| `other` | Прочее, что не подходит ни под один класс |

### Что делать аннотатору

1. Прочитайте `utterance_text` и (если нужно) учтите `speaker` и `cluster_name` диалога.
2. Проверьте, правильно ли проставлен `intent_label` (это автоматическая метка из блокнота 09).
3. Если метка верна — оставьте `intent_label_manual` пустым и поставьте `manual_status = ok`.
4. Если метка неверна — впишите правильный класс из списка выше в `intent_label_manual` и поставьте `manual_status = fixed`.
5. Если реплика неоднозначна или плохо подходит ни под один класс — поставьте `is_ambiguous = True`, по возможности укажите наиболее подходящий класс в `intent_label_manual`.
6. По желанию оставьте свободный комментарий в `manual_comment`.

## Ячейка 3. Загрузка weak-label датасета

In [ ]:
# ячейка 3: загрузка weak-label датасета
if WEAK_PARQUET.exists():
    df_weak = pd.read_parquet(WEAK_PARQUET)
    print(f"Загружен parquet: {WEAK_PARQUET}")
elif WEAK_CSV.exists():
    df_weak = pd.read_csv(WEAK_CSV)
    print(f"Загружен CSV (fallback): {WEAK_CSV}")
else:
    raise FileNotFoundError(
        "Не найден weak-label датасет. Ожидался один из файлов:\n"
        f"  - {WEAK_PARQUET}\n"
        f"  - {WEAK_CSV}\n"
        "Сначала запустите блокнот 09_intent_modeling_dialogsum_ru.ipynb."
    )

# Проверка обязательных колонок
REQUIRED_COLUMNS = [
    "utterance_id",
    "dialogue_id",
    "cluster_id",
    "cluster_name",
    "speaker",
    "utterance_text",
    "intent_label",
]
missing = [c for c in REQUIRED_COLUMNS if c not in df_weak.columns]
if missing:
    raise ValueError(
        "В weak-label датасете отсутствуют обязательные колонки: "
        f"{missing}.\nДоступные колонки: {list(df_weak.columns)}"
    )

print("\nShape :", df_weak.shape)
print("\nПервые строки:")
display(df_weak.head())

print("\nРаспределение intent_label:")
display(
    df_weak["intent_label"].value_counts(dropna=False).rename_axis("intent_label").to_frame("count")
)

print("\nРаспределение cluster_name (топ-20):")
display(
    df_weak["cluster_name"].value_counts(dropna=False).head(20).rename_axis("cluster_name").to_frame("count")
)

## Ячейка 4. Параметры сэмплирования и balanced sample

Идея сэмплирования:

- Для каждого intent-класса берём не более `MAX_PER_INTENT` примеров.
- Редкие классы (где доступных примеров меньше `MIN_PER_INTENT`) забираем **целиком** (`RARE_CLASS_KEEP_ALL = True`).
- Если итоговый размер превышает `REVIEW_TARGET_TOTAL`, делаем стратифицированное уменьшение, сохраняя редкие классы насколько возможно.

In [ ]:
# ячейка 4: параметры сэмплирования и формирование balanced sample
REVIEW_TARGET_TOTAL = 500
MIN_PER_INTENT = 20
MAX_PER_INTENT = 60
RARE_CLASS_KEEP_ALL = True

# Полный список 15 классов в каноническом порядке
INTENT_CLASSES = [
    "greeting",
    "thanks",
    "farewell",
    "informational_request",
    "clarification_request",
    "service_request",
    "purchase_or_booking_request",
    "complaint",
    "problem_report",
    "arrangement",
    "confirmation",
    "rejection",
    "suggestion_or_recommendation",
    "opinion_or_preference",
    "other",
]

# Чистим заведомо невалидные строки
df_pool = df_weak.copy()
df_pool = df_pool.dropna(subset=["utterance_text", "intent_label"])
df_pool = df_pool[df_pool["utterance_text"].astype(str).str.strip() != ""]
df_pool = df_pool.drop_duplicates(subset=["utterance_id"])

print("Размер пула после очистки:", df_pool.shape)

# Помечаем редкие классы относительно MIN_PER_INTENT
class_counts = df_pool["intent_label"].value_counts().to_dict()
rare_classes = {c for c, n in class_counts.items() if n < MIN_PER_INTENT}
print("\nЧастоты классов в пуле:")
for c in INTENT_CLASSES:
    print(f"  {c:32s} -> {class_counts.get(c, 0)}")
print("\nРедкие классы (n < MIN_PER_INTENT):", sorted(rare_classes))

# Шаг 1: для каждого класса берём до MAX_PER_INTENT случайных примеров
samples = []
for label in INTENT_CLASSES:
    sub = df_pool[df_pool["intent_label"] == label]
    if sub.empty:
        continue
    if RARE_CLASS_KEEP_ALL and label in rare_classes:
        chosen = sub  # редкий класс — берём весь
    else:
        n_take = min(MAX_PER_INTENT, len(sub))
        chosen = sub.sample(n=n_take, random_state=SEED)
    samples.append(chosen)

sample_df = pd.concat(samples, axis=0, ignore_index=True)
print("\nРазмер после первого шага сэмплирования:", sample_df.shape)

# Шаг 2: при необходимости стратифицированно уменьшаем до REVIEW_TARGET_TOTAL,
# сохраняя редкие классы полностью.
if len(sample_df) > REVIEW_TARGET_TOTAL:
    rare_part = sample_df[sample_df["intent_label"].isin(rare_classes)]
    common_part = sample_df[~sample_df["intent_label"].isin(rare_classes)]
    remaining_budget = max(REVIEW_TARGET_TOTAL - len(rare_part), 0)

    if remaining_budget == 0:
        sample_df = rare_part.reset_index(drop=True)
    else:
        # Стратифицированно по intent_label среди "частых" классов
        common_counts = common_part["intent_label"].value_counts()
        total_common = int(common_counts.sum())
        quotas = {}
        for label, n in common_counts.items():
            quotas[label] = int(round(remaining_budget * n / total_common))

        # Корректируем сумму квот до remaining_budget
        diff = remaining_budget - sum(quotas.values())
        if diff != 0 and quotas:
            labels_sorted = sorted(quotas, key=lambda x: -common_counts[x])
            i = 0
            while diff != 0 and labels_sorted:
                lab = labels_sorted[i % len(labels_sorted)]
                if diff > 0:
                    if quotas[lab] < common_counts[lab]:
                        quotas[lab] += 1
                        diff -= 1
                else:
                    if quotas[lab] > 0:
                        quotas[lab] -= 1
                        diff += 1
                i += 1
                if i > 10_000:
                    break

        common_chunks = []
        for label, q in quotas.items():
            sub = common_part[common_part["intent_label"] == label]
            q = min(q, len(sub))
            if q > 0:
                common_chunks.append(sub.sample(n=q, random_state=SEED))
        common_reduced = (
            pd.concat(common_chunks, axis=0, ignore_index=True) if common_chunks else common_part.iloc[0:0]
        )
        sample_df = pd.concat([rare_part, common_reduced], axis=0, ignore_index=True)

# Финальное перемешивание
sample_df = sample_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print("\nИтоговый размер sample:", sample_df.shape)
print("\nИтоговое распределение по intent_label:")
display(
    sample_df["intent_label"].value_counts().rename_axis("intent_label").to_frame("count")
)

## Ячейка 5. Добавление полей для ручной проверки

In [ ]:
# ячейка 5: добавление полей для ручной проверки
review_df = sample_df.copy()

# Базовые поля разметки
review_df["intent_label_manual"] = ""
review_df["manual_status"] = "pending"
review_df["manual_comment"] = ""
review_df["is_ambiguous"] = False

# Классы, которые сложнее всего размечать слабыми правилами — повышенный приоритет
HARD_CLASSES = {
    "other",
    "complaint",
    "problem_report",
    "arrangement",
    "opinion_or_preference",
    "suggestion_or_recommendation",
}

# Опциональная колонка с уверенностью weak-label (если есть в исходном датасете)
confidence_col = None
for candidate in ["intent_confidence", "confidence", "score", "weak_label_score"]:
    if candidate in review_df.columns:
        confidence_col = candidate
        break

LOW_CONF_THRESHOLD = 0.5

def _priority(row):
    label = row["intent_label"]
    if label in rare_classes:
        return "high"
    if confidence_col is not None:
        try:
            if float(row[confidence_col]) < LOW_CONF_THRESHOLD:
                return "high"
        except (TypeError, ValueError):
            pass
    if label in HARD_CLASSES:
        return "high"
    return "normal"

review_df["review_priority"] = review_df.apply(_priority, axis=1)

review_df["annotation_instruction"] = (
    "Проверьте, соответствует ли intent_label содержанию реплики. "
    "Если соответствует — поставьте manual_status = ok. "
    "Если нет — впишите правильный класс из codebook в intent_label_manual "
    "и поставьте manual_status = fixed."
)

# Упорядочим колонки для удобства человека
leading_cols = [
    "utterance_id",
    "dialogue_id",
    "speaker",
    "cluster_id",
    "cluster_name",
    "utterance_text",
    "intent_label",
    "intent_label_manual",
    "manual_status",
    "is_ambiguous",
    "manual_comment",
    "review_priority",
    "annotation_instruction",
]
other_cols = [c for c in review_df.columns if c not in leading_cols]
review_df = review_df[leading_cols + other_cols]

print("Колонка с confidence:", confidence_col)
print("Размер review_df:", review_df.shape)
print("\nРаспределение review_priority:")
display(review_df["review_priority"].value_counts().to_frame("count"))
print("\nПервые строки:")
display(review_df.head())

## Ячейка 6. Контроль качества выборки

In [ ]:
# ячейка 6: контроль качества sample
print("Итоговый размер выборки:", len(review_df))

print("\nРаспределение по intent_label:")
display(
    review_df["intent_label"].value_counts().rename_axis("intent_label").to_frame("count")
)

print("\nРаспределение по cluster_name (топ-20):")
display(
    review_df["cluster_name"].value_counts().head(20).rename_axis("cluster_name").to_frame("count")
)

print("\nРаспределение по speaker:")
display(review_df["speaker"].value_counts().to_frame("count"))

# Проверка дубликатов
dup_mask = review_df["utterance_id"].duplicated(keep=False)
n_dup = int(dup_mask.sum())
print(f"\nДубликатов по utterance_id: {n_dup}")
if n_dup > 0:
    display(review_df.loc[dup_mask].sort_values("utterance_id").head(20))

# Пустые тексты
empty_mask = review_df["utterance_text"].astype(str).str.strip() == ""
n_empty = int(empty_mask.sum())
print(f"Пустых utterance_text: {n_empty}")

# 10 случайных строк для визуальной проверки
print("\n10 случайных строк:")
display(review_df.sample(n=min(10, len(review_df)), random_state=SEED))

## Ячейка 7. Сохранение файлов CSV и Parquet

In [ ]:
# ячейка 7: сохранить CSV/Parquet
# CSV: utf-8-sig — корректно открывается в Excel/Google Sheets с кириллицей
review_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
review_df.to_parquet(OUT_PARQUET, index=False)

print("Сохранены файлы для ручной валидации:")
print("  CSV     :", OUT_CSV)
print("  Parquet :", OUT_PARQUET)
print("\nРазмер CSV (байт)    :", OUT_CSV.stat().st_size if OUT_CSV.exists() else "—")
print("Размер Parquet (байт):", OUT_PARQUET.stat().st_size if OUT_PARQUET.exists() else "—")

## Ячейка 8. Codebook для аннотатора

Программно формируем markdown-файл с инструкцией, описанием 15 классов и описанием полей таблицы.

In [ ]:
# ячейка 8: создать codebook markdown
INTENT_DESCRIPTIONS = [
    ("greeting", "приветствие"),
    ("thanks", "благодарность"),
    ("farewell", "завершение разговора"),
    ("informational_request", "запрос информации"),
    ("clarification_request", "уточняющий вопрос"),
    ("service_request", "просьба выполнить действие/услугу"),
    ("purchase_or_booking_request", "покупка, заказ или бронирование"),
    ("complaint", "жалоба/недовольство"),
    ("problem_report", "сообщение о проблеме"),
    ("arrangement", "договорённость о времени/месте/условиях"),
    ("confirmation", "подтверждение/согласие"),
    ("rejection", "отказ/несогласие"),
    ("suggestion_or_recommendation", "совет/рекомендация/предложение"),
    ("opinion_or_preference", "мнение/предпочтение"),
    ("other", "прочее"),
]

FIELD_DESCRIPTIONS = [
    ("utterance_id", "Уникальный идентификатор реплики. НЕ изменять."),
    ("dialogue_id", "Идентификатор диалога, к которому относится реплика. НЕ изменять."),
    ("speaker", "Говорящий (#Person1#, #Person2# и т. д.). НЕ изменять."),
    ("cluster_id", "Идентификатор тематического кластера диалога (из блокнота 08). Справочно."),
    ("cluster_name", "Название тематического кластера. Справочно — помогает понять контекст."),
    ("utterance_text", "Текст реплики на русском. Это основной объект для разметки."),
    ("intent_label", "Автоматическая weak-label метка из блокнота 09. Может быть ошибочной."),
    ("intent_label_manual", "Поле для аннотатора. Если intent_label верен — оставить пустым. Если нет — вписать правильный класс из списка ниже."),
    ("manual_status", "Статус: `pending` (по умолчанию), `ok` (метка верна), `fixed` (метка исправлена в intent_label_manual), `skip` (нельзя разметить)."),
    ("is_ambiguous", "True/False. Поставить True, если реплика неоднозначна и подходит под несколько классов."),
    ("manual_comment", "Свободный комментарий аннотатора (необязательно)."),
    ("review_priority", "Приоритет проверки: `high` (редкие/сложные классы, низкая уверенность) или `normal`. Справочно."),
    ("annotation_instruction", "Краткая подсказка для аннотатора. Справочно."),
]

lines = []
lines.append("# Codebook ручной валидации intent-разметки DialogSum-RU\n")
lines.append("Этот файл сопровождает таблицу "
             "`dialogsum_ru_intent_manual_review.csv` / `.parquet` "
             "и описывает правила ручной разметки.\n")

lines.append("## Что нужно сделать аннотатору\n")
lines.append(
    "1. Открыть CSV-файл в Google Sheets / Excel "
    "(он сохранён в кодировке UTF-8 с BOM, кириллица отображается корректно).\n"
    "2. Прочитать колонку `utterance_text`. При необходимости — учесть контекст через "
    "`speaker` и `cluster_name`.\n"
    "3. Проверить колонку `intent_label` (это автоматическая weak-label из блокнота 09).\n"
    "4. Если метка **верна** — оставить `intent_label_manual` пустым и "
    "поставить `manual_status = ok`.\n"
    "5. Если метка **неверна** — вписать правильный класс из списка ниже в "
    "`intent_label_manual` и поставить `manual_status = fixed`.\n"
    "6. Если реплика **неоднозначна** — `is_ambiguous = True`, по возможности указать "
    "наиболее подходящий класс.\n"
    "7. Если реплику **невозможно разметить** — `manual_status = skip` и комментарий в "
    "`manual_comment`.\n"
    "8. Свободный комментарий по желанию — в `manual_comment`.\n"
)

lines.append("## 15 классов намерений\n")
lines.append("| # | Класс | Описание |")
lines.append("|---|---|---|")
for i, (code, desc) in enumerate(INTENT_DESCRIPTIONS, start=1):
    lines.append(f"| {i} | `{code}` | {desc} |")
lines.append("")

lines.append("## Описание полей таблицы\n")
lines.append("| Поле | Назначение |")
lines.append("|---|---|")
for name, desc in FIELD_DESCRIPTIONS:
    lines.append(f"| `{name}` | {desc} |")
lines.append("")

lines.append("## Допустимые значения\n")
lines.append("- `manual_status` ∈ {`pending`, `ok`, `fixed`, `skip`}.\n"
             "- `is_ambiguous` ∈ {True, False}.\n"
             "- `intent_label_manual` ∈ {список 15 классов выше} либо пусто.\n")

lines.append("## Принципы разметки\n")
lines.append(
    "- Размечается **речевой акт реплики** (что делает говорящий), а не тема разговора.\n"
    "- Одна реплика — **один** intent-класс. Если их несколько, выбрать доминирующий.\n"
    "- Класс `other` использовать только если ни один из 14 содержательных классов не подходит.\n"
    "- При сомнениях — `is_ambiguous = True` и кратко описать сомнение в `manual_comment`.\n"
)

codebook_text = "\n".join(lines)
OUT_CODEBOOK.write_text(codebook_text, encoding="utf-8")

print("Codebook сохранён:", OUT_CODEBOOK)
print("\nПервые 800 символов:\n")
print(codebook_text[:800])

## Дальнейшие шаги

После того как аннотатор заполнит CSV вручную:

1. Загрузить размеченный файл обратно (`pd.read_csv(..., encoding="utf-8-sig")`).
2. Построить итоговый **gold label** по правилу: если `manual_status = ok` — использовать `intent_label`; если `manual_status = fixed` — использовать `intent_label_manual`; строки со `skip` исключить.
3. Сравнить распределения weak vs gold; посчитать **долю согласия** weak-label с gold по каждому классу.
4. Переоценить baseline-модели из блокнота 09 на этом ручном подмножестве — это даст более честную оценку качества, чем оценка на weak-label.
5. При желании — использовать подмножество как небольшое **golden test-set** для дальнейших экспериментов с pretrained-моделями.